# Pipeline C — Step 1: Feature Engineering (Pseudo-MIDAS)

**Filosofi:** Menggunakan data bulanan (1488 obs) tanpa agregasi tahunan.
Fitur tahunan di-*forward-fill*, sementara fitur bulanan (BI Rate, Inflasi, TWP90 lag) tetap murni.
Pendekatan Pseudo-MIDAS: biarkan XGBoost menemukan pembobotan waktu secara otomatis.

**Input:** `model_panel.csv` (root)

**Output:** `pipeline_C/output/C1_monthly_features.csv`

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# --- Locate project root ---
def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for p in [start, *start.parents]:
        if (p / 'model_panel.csv').exists():
            return p
    raise FileNotFoundError('Could not find model_panel.csv')

ROOT = find_project_root(Path.cwd())
OUT  = ROOT / 'pipeline_C' / 'output'
OUT.mkdir(parents=True, exist_ok=True)
print(f'Project root: {ROOT}')

Project root: C:\Users\bimyu\Documents\Projects\DatathonMETC


## 1. Load & Clean Raw Data

In [2]:
df = pd.read_csv(ROOT / 'model_panel.csv')

# Remove provinsi_id == 19 (consistent with Pipeline A)
df = df[df['provinsi_id'] != 19].copy()

# Keep only 2022-2025 (consistent with Pipeline A)
df = df[df['tahun'] > 2021].copy()

# Sort for proper lag computation
df = df.sort_values(['provinsi_id', 'tahun', 'bulan']).reset_index(drop=True)

print(f'Shape: {df.shape}')
print(f'Provinces: {df["provinsi_id"].nunique()}')
print(f'Date range: {df["tahun"].min()}-{df["bulan"].min():02d} to {df["tahun"].max()}-{df["bulan"].max():02d}')
df.head()

Shape: (1488, 16)
Provinces: 31
Date range: 2022-01 to 2025-12


,provinsi_id,nama_provinsi,tanggal,tahun,bulan,twp90_pct,x1_bi_rate_pct,x2_inflasi_yoy,x3_pdrb_per_kapita,x4_tpt_pct,x5_penetrasi_internet_pct,x6_tabungan_miliar,x7_jumlah_kc_bank,x8_ldr_pct,x9_npl_ratio,x10_rasio_umkm
0,1,Banten,2022-01-01,2022,1,0.0217,3.5,0.00810,61414000.0,NaN,0.81,236586.03,96.0,0.7703,0.0202,0.2979
1,1,Banten,2022-02-01,2022,2,0.0217,3.5,0.00017,61414000.0,8.53,0.81,236586.03,96.0,0.7703,0.0202,0.2979
2,1,Banten,2022-03-01,2022,3,0.0225,3.5,0.01087,61414000.0,NaN,0.81,236586.03,96.0,0.7703,0.0202,0.2979
3,1,Banten,2022-04-01,2022,4,0.0238,3.5,0.00973,61414000.0,NaN,0.81,236586.03,96.0,0.7703,0.0202,0.2979
4,1,Banten,2022-05-01,2022,5,0.0255,3.5,0.00383,61414000.0,NaN,0.81,236586.03,96.0,0.7703,0.0202,0.2979


## 2. Handle Missing Values

In [3]:
print('=== Missing values BEFORE imputation ===')
print(df.isnull().sum())
print()

# x4_tpt_pct (unemployment) - only available in Feb & Aug (BPS schedule)
# Forward-fill within each province (carry last known value)
df['x4_tpt_pct'] = df.groupby('provinsi_id')['x4_tpt_pct'].ffill()
df['x4_tpt_pct'] = df.groupby('provinsi_id')['x4_tpt_pct'].bfill()

# x5_penetrasi_internet_pct - annual, forward-filled
df['x5_penetrasi_internet_pct'] = df.groupby('provinsi_id')['x5_penetrasi_internet_pct'].ffill()
df['x5_penetrasi_internet_pct'] = df.groupby('provinsi_id')['x5_penetrasi_internet_pct'].bfill()

print('=== Missing values AFTER imputation ===')
print(df.isnull().sum())

=== Missing values BEFORE imputation ===
provinsi_id                     0
nama_provinsi                   0
tanggal                         0
tahun                           0
bulan                           0
twp90_pct                       0
x1_bi_rate_pct                  0
x2_inflasi_yoy                  0
x3_pdrb_per_kapita              0
x4_tpt_pct                   1240
x5_penetrasi_internet_pct       0
x6_tabungan_miliar              0
x7_jumlah_kc_bank               0
x8_ldr_pct                      0
x9_npl_ratio                    0
x10_rasio_umkm                  0
dtype: int64

=== Missing values AFTER imputation ===
provinsi_id                  0
nama_provinsi                0
tanggal                      0
tahun                        0
bulan                        0
twp90_pct                    0
x1_bi_rate_pct               0
x2_inflasi_yoy               0
x3_pdrb_per_kapita           0
x4_tpt_pct                   0
x5_penetrasi_internet_pct    0
x6_tabungan_miliar  

## 3. Feature Engineering

### 3.1 Autoregressive Lags (TWP90)

In [4]:
# Autoregressive lags on TWP90 (the most powerful feature for monthly prediction)
for lag in [1, 2, 3, 6, 12]:
    df[f'twp90_lag_{lag}'] = df.groupby('provinsi_id')['twp90_pct'].shift(lag)

# Rolling statistics (momentum indicators)
df['twp90_roll3_mean'] = df.groupby('provinsi_id')['twp90_pct'].transform(
    lambda s: s.rolling(3, min_periods=1).mean()
)
df['twp90_roll3_std']  = df.groupby('provinsi_id')['twp90_pct'].transform(
    lambda s: s.rolling(3, min_periods=1).std()
)
df['twp90_roll3_std'] = df['twp90_roll3_std'].fillna(0)

# Month-over-month change
df['twp90_mom'] = df.groupby('provinsi_id')['twp90_pct'].diff()

print('AR features created:', [c for c in df.columns if 'twp90_lag' in c or 'twp90_roll' in c or 'twp90_mom' in c])

AR features created: ['twp90_lag_1', 'twp90_lag_2', 'twp90_lag_3', 'twp90_lag_6', 'twp90_lag_12', 'twp90_roll3_mean', 'twp90_roll3_std', 'twp90_mom']


### 3.2 Monthly Feature Lags (BI Rate, Inflation)

In [5]:
# Lagged monthly features (policy transmission delay)
for lag in [1, 3]:
    df[f'bi_rate_lag_{lag}'] = df.groupby('provinsi_id')['x1_bi_rate_pct'].shift(lag)
    df[f'inflasi_lag_{lag}'] = df.groupby('provinsi_id')['x2_inflasi_yoy'].shift(lag)

# Inflation momentum
df['inflasi_mom'] = df.groupby('provinsi_id')['x2_inflasi_yoy'].diff()

print('Monthly lag features created.')

Monthly lag features created.


### 3.3 Temporal & Panel Identity Features

In [6]:
# Cyclical encoding for month (captures seasonality without discontinuity)
df['bulan_sin'] = np.sin(2 * np.pi * df['bulan'] / 12)
df['bulan_cos'] = np.cos(2 * np.pi * df['bulan'] / 12)

# Linear time trend (global trend)
df['time_trend'] = (df['tahun'] - 2022) * 12 + df['bulan']

# Log-transform PDRB (same as Pipeline A)
df['log_pdrb'] = np.log(df['x3_pdrb_per_kapita'])

print('Temporal features created.')
print(f'time_trend range: {df["time_trend"].min()} to {df["time_trend"].max()}')

Temporal features created.
time_trend range: 1 to 48


### 3.4 Pseudo-MIDAS Interaction Features

Instead of formal MIDAS polynomial weighting, we create interaction terms
between annual-frequency variables and the month position within the year.
This lets XGBoost learn time-varying weights for annual proxy variables.

In [7]:
# Month position within year (0 to 1 scale)
month_weight = df['bulan'] / 12.0

# Annual proxy variables that are forward-carried within each year
annual_vars = ['log_pdrb', 'x5_penetrasi_internet_pct', 'x8_ldr_pct', 'x9_npl_ratio', 'x10_rasio_umkm']

for var in annual_vars:
    df[f'{var}_x_month'] = df[var] * month_weight

print('Pseudo-MIDAS interaction features created:')
print([c for c in df.columns if '_x_month' in c])

Pseudo-MIDAS interaction features created:
['log_pdrb_x_month', 'x5_penetrasi_internet_pct_x_month', 'x8_ldr_pct_x_month', 'x9_npl_ratio_x_month', 'x10_rasio_umkm_x_month']


## 4. Define Feature Set & Save

In [8]:
# === FINAL FEATURE SET ===
target = 'twp90_pct'

# Monthly-frequency features (truly monthly)
monthly_features = [
    'x1_bi_rate_pct', 'x2_inflasi_yoy',
    'bi_rate_lag_1', 'bi_rate_lag_3',
    'inflasi_lag_1', 'inflasi_lag_3', 'inflasi_mom',
]

# Autoregressive features
ar_features = [
    'twp90_lag_1', 'twp90_lag_2', 'twp90_lag_3', 'twp90_lag_6', 'twp90_lag_12',
    'twp90_roll3_mean', 'twp90_roll3_std', 'twp90_mom',
]

# Annual proxy features (forward-carried, NOT interpolated)
annual_proxy = [
    'log_pdrb', 'x4_tpt_pct', 'x5_penetrasi_internet_pct',
    'x6_tabungan_miliar', 'x7_jumlah_kc_bank',
    'x8_ldr_pct', 'x9_npl_ratio', 'x10_rasio_umkm',
]

# Pseudo-MIDAS interactions
midas_features = [f'{v}_x_month' for v in ['log_pdrb', 'x5_penetrasi_internet_pct', 'x8_ldr_pct', 'x9_npl_ratio', 'x10_rasio_umkm']]

# Temporal / panel identity
identity_features = ['provinsi_id', 'bulan_sin', 'bulan_cos', 'time_trend']

feature_cols = monthly_features + ar_features + annual_proxy + midas_features + identity_features

print(f'Total features: {len(feature_cols)}')
print(f'  Monthly:   {len(monthly_features)}')
print(f'  AR:        {len(ar_features)}')
print(f'  Annual:    {len(annual_proxy)}')
print(f'  MIDAS:     {len(midas_features)}')
print(f'  Identity:  {len(identity_features)}')

Total features: 32
  Monthly:   7
  AR:        8
  Annual:    8
  MIDAS:     5
  Identity:  4


In [9]:
# Drop rows with NaN (from lag computation — first 12 months per province)
keep_cols = [target] + feature_cols + ['nama_provinsi', 'tahun', 'bulan']
df_out = df[keep_cols].copy()

before = len(df_out)
df_out = df_out.dropna(subset=feature_cols + [target])
after = len(df_out)

print(f'Rows before dropna: {before}')
print(f'Rows after dropna:  {after}  (dropped {before - after} rows due to lags)')
print(f'Year range: {df_out["tahun"].min()} - {df_out["tahun"].max()}')
print(f'Provinces:  {df_out["provinsi_id"].nunique()}')

Rows before dropna: 1488
Rows after dropna:  1116  (dropped 372 rows due to lags)
Year range: 2023 - 2025
Provinces:  31


In [10]:
# Save feature matrix
out_path = OUT / 'C1_monthly_features.csv'
df_out.to_csv(out_path, index=False)
print(f'Saved: {out_path}')

# Save feature column list for C2
import json
meta = {'target': target, 'feature_cols': feature_cols}
with open(OUT / 'C1_feature_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)
print(f'Feature metadata saved.')

Saved: C:\Users\bimyu\Documents\Projects\DatathonMETC\pipeline_C\output\C1_monthly_features.csv
Feature metadata saved.
